In [3]:
from pydantic import Field, BaseModel, field_validator
from decimal import Decimal
from langchain_core.tools import tool
#流式输出 stream_events
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
#1.读取环境变量
load_dotenv()

# 1. 定义工具参数 Schema
class TransferInput(BaseModel):
    account: str = Field(
        description="收款银行卡号，必须为16到19位数字",
        min_length=16,
        max_length=19
    )

    amount: Decimal = Field(
        description="转账金额，单位元，必须大于0且不超过50000",
        gt=0,
        le=50000,
        max_digits=10,
        decimal_places=2
    )

    remark: str = Field(
        default="",
        description="转账备注，最多100个字符",
        max_length=100
    )

    @field_validator("account")
    @classmethod
    def validate_account(cls, value: str) -> str:
        if not value.isascii() or not value.isdigit():
            raise ValueError("银行卡号必须为纯数字")
        return value


# 2. 定义工具函数
@tool(
    description="模拟银行转账，接收银行卡号、金额和备注",
    args_schema=TransferInput
)
def bank_transfer(
    account: str,
    amount: Decimal,
    remark: str = ""
) -> str:
    print("\n[工具执行] bank_transfer")
    print(f"收款账户: {account}")
    print(f"金额: {amount}")

    return (
        f"模拟转账成功！账户：{account}，"
        f"金额：{amount}元，备注：{remark}"
    )
# 3. 配置参数校验异常处理
# 校验失败时，将错误信息反馈给 Agent
bank_transfer.handle_validation_error = (
    lambda e: f"转账参数校验失败：{e}。请检查参数。"
)
tools = [bank_transfer]
#2.初始化模型
model = init_chat_model(model="deepseek-v4-flash",extra_body={"thinking":{"type":"disabled"}})
agent = create_agent(
    model=model,
    system_prompt=(
        "你是一名银行业务助手。"
        "用户请求转账时，先检查信息是否完整。"
        "账户、金额缺失时应询问用户，不能编造。"
        "仅当用户明确要求执行模拟转账时才调用工具。"
        "工具参数校验失败时，解释错误并要求用户修正。"
        "不要擅自修改用户提供的账号或金额。"
    ),
    tools=tools
)

stream_response = agent.stream_events(input={"messages":[HumanMessage("帮我向622转账1000元，备注生活费")]},version="v3")
for message in stream_response.messages:
    for chunk in message.text:
        print(chunk,end="",flush=True)

您提供的收款账号“622”不完整——银行卡号必须为16到19位数字。请补充完整的收款卡号，我再为您处理这笔1000元、备注“生活费”的模拟转账。